# pandas 00. まずはこの10個

pandas にはメソッドが数百ありますが、普段書くのはだいたいこの10個です。

| # | やること | SQL でいうと | pandas |
| --- | --- | --- | --- |
| 1 | 読む | — | `pd.read_csv` |
| 2 | 見る | — | `head` / `shape` / `dtypes` / `value_counts` |
| 3 | 列を選ぶ | `SELECT` | `df[["a", "b"]]` |
| 4 | 行を絞る | `WHERE` | `df[条件]` |
| 5 | 列を作る | `SELECT ... AS` | `df["new"] = ...` |
| 6 | 並べる | `ORDER BY` | `sort_values` |
| 7 | まとめる | `GROUP BY` | `groupby().agg()` |
| 8 | 横にくっつける | `JOIN` | `merge` |
| 9 | 縦にくっつける | `UNION ALL` | `concat` |
| 10 | 重複を消す | `DISTINCT` | `drop_duplicates` |

最後に「書く」(`to_parquet`)が付いて終わりです。

## このノートの進み方

**説明 → セルを実行 → ミニ練習を1問**、の繰り返しです。

ミニ練習は「列の名前を変えるだけ」「条件を1つ変えるだけ」くらいの軽さにしてあります。
`assert` が通れば `OK` と出ます。分からなければ、すぐ下の**答え**を開いてしまって
かまいません。写して動かすだけでも十分です。

使うデータは `/data/sales.csv`(12行)と `/data/shops.csv`(4行)。
小さいので、結果を目で数えて確かめられます。

---
## 0. 出てくる型は2つだけ

pandas のデータは、**表**か**1列**か、のどちらかです。

In [ ]:
import pandas as pd

# DataFrame = 表。Excel のシート1枚のようなもの
df = pd.read_csv("/data/sales.csv")
df

In [ ]:
# Series = 1列。表から1列だけ取り出したもの
s = df["qty"]
print(type(s).__name__)
s

| | 何か | 取り出し方 |
| --- | --- | --- |
| **DataFrame** | 表(2次元) | `pd.read_csv(...)` |
| **Series** | 1列(1次元) | `df["列名"]` |

角括弧を2つにすると、1列でも**表のまま**取り出せます。

```python
df["qty"]      # Series
df[["qty"]]    # DataFrame (列が1つの表)
```

左端の `0 1 2 3 ...` は **index**(行の名前)です。いまは連番ですが、
行を絞ったり並べ替えたりすると飛び番になります。気になったら
`.reset_index(drop=True)` で振り直せます。

In [ ]:
# ミニ練習: price 列を Series として取り出す

ans = ...   # ここに書く

assert type(ans).__name__ == "Series"
assert len(ans) == 12
print("OK")

<details>
<summary>答え</summary>

```python
ans = df["price"]
```

</details>

---
## 1. 読む

In [ ]:
df = pd.read_csv("/data/sales.csv")
shops = pd.read_csv("/data/shops.csv")

display(df.head(3))
shops

```python
pd.read_csv(path)        # CSV
pd.read_parquet(path)    # Parquet
pd.read_excel(path)      # Excel
pd.read_json(path)       # JSON
```

だいたい `pd.read_なんとか(パス)` で読めます。

引数はたくさんありますが、最初は何も付けなくて大丈夫です。
付けたほうがよい引数の話は `pandas-01` でやります。

In [ ]:
# ミニ練習: shops.csv を読んで shops2 という名前にする

shops2 = ...   # ここに書く

assert list(shops2.columns) == ["shop", "area", "manager"]
assert len(shops2) == 4
print("OK")

<details>
<summary>答え</summary>

```python
shops2 = pd.read_csv("/data/shops.csv")
```

</details>

---
## 2. 見る

何をするにも、まずこれです。データを見ずに書き始めると、たいてい外します。

In [ ]:
print("行数と列数:", df.shape)      # (行, 列)
print("列名:", df.columns.tolist())
print()
print("型:")
print(df.dtypes)

In [ ]:
display(df.head(3))      # 最初の3行
display(df.tail(3))      # 最後の3行
df.sample(3)             # ランダムに3行

In [ ]:
# この列にどんな値が入っているか
print("種類:", df["shop"].unique())
print("種類数:", df["shop"].nunique())
print()
print("値ごとの件数:")
print(df["shop"].value_counts())

| やりたいこと | 書き方 |
| --- | --- |
| 大きさ | `df.shape` |
| 列名 | `df.columns.tolist()` |
| 型 | `df.dtypes` |
| 最初/最後を見る | `df.head()` / `df.tail()` |
| **どんな値があるか** | **`df["col"].unique()`** |
| 値ごとの件数 | `df["col"].value_counts()` |
| 欠損の数 | `df.isna().sum()` |
| 数値の要約 | `df.describe()` |

`unique()` と `value_counts()` を最初に叩く癖をつけると楽です。
仕様書を読んで想像するより、データに聞いたほうが速くて確実なので。

In [ ]:
# ミニ練習: item 列の「値ごとの件数」を出す

ans = ...   # ここに書く

assert ans["コーヒー"] == 5
assert ans["ケーキ"] == 4
assert ans["紅茶"] == 3
print("OK")

<details>
<summary>答え</summary>

```python
ans = df["item"].value_counts()
```

</details>

---
## 3. 列を選ぶ (SELECT)

In [ ]:
df[["shop", "item", "qty"]]

```sql
SELECT shop, item, qty FROM sales
```

リストで渡した**順に並びます**。並べ替えも同時にできます。

```python
df[["qty", "shop"]]     # この順になる
```

逆に、いらない列を捨てる書き方もあります。

```python
df.drop(columns=["price"])      # price 以外
```

残す列が少なければ `df[[...]]`、捨てる列が少なければ `drop`、くらいの使い分けで十分です。

In [ ]:
# ミニ練習: sale_id, item, price の3列を、この順で取り出す

ans = ...   # ここに書く

assert list(ans.columns) == ["sale_id", "item", "price"]
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[["sale_id", "item", "price"]]
```

</details>

---
## 4. 行を絞る (WHERE)

In [ ]:
df[df["shop"] == "渋谷"]

In [ ]:
# 中で何が起きているか。条件式は True/False の Series になる
mask = df["shop"] == "渋谷"
print(mask.tolist())
print()
print("これを [] に渡すと、True の行だけ残る")
df[mask]

In [ ]:
# 条件を複数つなぐ。& (かつ) | (または) ~ (でない)
# それぞれの条件を括弧で囲む
df[(df["shop"] == "渋谷") & (df["qty"] >= 2)]

In [ ]:
# よく使う書き方
display(df[df["shop"].isin(["渋谷", "横浜"])])   # IN
display(df[df["qty"].between(2, 3)])             # BETWEEN
df[df["item"].str.contains("コーヒー")]           # LIKE '%コーヒー%'

```sql
SELECT * FROM sales WHERE shop = '渋谷' AND qty >= 2
```

| SQL | pandas |
| --- | --- |
| `AND` | `&` |
| `OR` | `\|` |
| `NOT` | `~` |
| `IN (...)` | `.isin([...])` |
| `BETWEEN a AND b` | `.between(a, b)` |
| `LIKE '%x%'` | `.str.contains("x")` |

ひとつだけ注意点です。**`and` `or` `not` は使えません。**
`&` `|` `~` を使い、**それぞれの条件を括弧で囲みます**。
忘れると分かりにくいエラーが出ますが、原因はいつもこれです。

In [ ]:
# ミニ練習: shop が "新宿" の行だけを取り出す

ans = ...   # ここに書く

assert len(ans) == 4, f"4行のはず: {len(ans)}"
assert set(ans["sale_id"]) == {"S-03", "S-06", "S-09", "S-12"}
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[df["shop"] == "新宿"]
```

</details>

In [ ]:
# ミニ練習: item が "ケーキ" で、かつ qty が 2 以上の行を取り出す
#           (条件は括弧で囲む)

ans = ...   # ここに書く

assert len(ans) == 2, f"2行のはず: {len(ans)}"
assert set(ans["sale_id"]) == {"S-05", "S-10"}
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[(df["item"] == "ケーキ") & (df["qty"] >= 2)]
```

</details>

---
## 5. 列を作る (SELECT ... AS)

In [ ]:
df["total"] = df["qty"] * df["price"]
df.head(4)

```sql
SELECT *, qty * price AS total FROM sales
```

列どうしの計算は、**行ごとに勝手に対応**してくれます。ループは要りません。
これを**ベクトル化**と呼びます。pandas でループを書きたくなったら、
たいてい書かずに済む方法があります。

In [ ]:
# 文字列の列も同じ。.str を挟むと文字列の操作ができる
df["label"] = df["shop"] + "/" + df["item"]
df[["sale_id", "label"]].head(4)

In [ ]:
# 条件で値を分ける
import numpy as np
df["size"] = np.where(df["qty"] >= 3, "大口", "通常")
df[["sale_id", "qty", "size"]].head(4)

In [ ]:
# ミニ練習: total の 10% を tax という列にして df に足す

# ここに書く

assert "tax" in df.columns
assert round(df["tax"].sum(), 6) == 1120.0, f"合計が違う: {df['tax'].sum()}"
print("OK")

<details>
<summary>答え</summary>

```python
df["tax"] = df["total"] * 0.1
```

</details>

---
## 6. 並べる (ORDER BY)

In [ ]:
df.sort_values("total", ascending=False).head(5)

In [ ]:
# 複数キー。向きも列ごとに変えられる
df.sort_values(["shop", "total"], ascending=[True, False]).head(5)

```sql
SELECT * FROM sales ORDER BY shop ASC, total DESC
```

- `ascending=True` が既定(昇順)です
- 複数列はリストで渡します。`ascending` もリストで渡せます
- **元の `df` は変わりません。** 並べ替えた新しい表が返ります

index が飛び番になるのが気になるなら `.reset_index(drop=True)` を足します。

In [ ]:
# ミニ練習: qty の多い順に並べて、いちばん上の行の sale_id を取り出す

ans = ...   # ここに書く

assert ans == "S-07", f"S-07 のはず: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.sort_values("qty", ascending=False)["sale_id"].iloc[0]
```

</details>

---
## 7. まとめる (GROUP BY)

pandas でいちばんよく使う操作です。

In [ ]:
df.groupby("shop", as_index=False).agg(
    n_sales=("sale_id", "size"),
    total=("total", "sum"),
)

```sql
SELECT shop, count(*) AS n_sales, sum(total) AS total
FROM sales GROUP BY shop
```

書き方はこの形で覚えてしまうのが早いです。

```python
df.groupby(キー, as_index=False).agg(
    出したい列名=(元の列, 集約関数),
    ...
)
```

| 集約関数 | 意味 |
| --- | --- |
| `"size"` | 行数 (`count(*)`) |
| `"count"` | 欠損でない数 (`count(col)`) |
| `"sum"` | 合計 |
| `"mean"` | 平均 |
| `"min"` / `"max"` | 最小 / 最大 |
| `"nunique"` | 種類数 (`count(distinct)`) |

`as_index=False` を付けると、キーが普通の列として残ります。
付けないとキーが index に入って少し扱いにくいので、とりあえず付けておきましょう。

In [ ]:
# キーは複数指定できる
df.groupby(["shop", "item"], as_index=False).agg(
    qty=("qty", "sum"),
    total=("total", "sum"),
).head(5)

In [ ]:
# 1列だけでよければ短く書ける
df.groupby("item")["total"].sum()

In [ ]:
# ミニ練習: item ごとの qty の合計を出す (キーは列として残す)

ans = ...   # ここに書く

assert list(ans.columns) == ["item", "qty"], list(ans.columns)
assert ans.set_index("item")["qty"].to_dict() == {"コーヒー": 12, "ケーキ": 7, "紅茶": 4}
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("item", as_index=False).agg(qty=("qty", "sum"))
```

</details>

---
## 8. 横にくっつける (JOIN)

In [ ]:
display(shops)
df.merge(shops, on="shop", how="left").head(4)

```sql
SELECT * FROM sales LEFT JOIN shops ON sales.shop = shops.shop
```

```python
左.merge(右, on=キー, how=結合の種類)
```

| `how` | 意味 |
| --- | --- |
| `"inner"` | 両方にある行だけ(既定) |
| `"left"` | **左を全部残す**。よく使う |
| `"right"` | 右を全部残す |
| `"outer"` | 両方を全部残す |

キーの列名が左右で違うときは `left_on` / `right_on` を使います。

結合したら、**行数を見る**のを習慣にしてください。右側にキーの重複があると
行が増えますし、`inner` だと結合できなかった行が黙って消えます。

In [ ]:
# 行数が変わっていないかを確かめる
merged = df.merge(shops, on="shop", how="left")
print("結合前:", len(df))
print("結合後:", len(merged))

# 大宮は売上が無いので、left join では出てこない
print("結合できなかった行:", merged["area"].isna().sum())

In [ ]:
# ミニ練習: shops と結合して、area が "神奈川" の行だけ取り出す

ans = ...   # ここに書く

assert len(ans) == 3, f"3行のはず: {len(ans)}"
assert set(ans["shop"]) == {"横浜"}
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.merge(shops, on="shop", how="left")
ans = ans[ans["area"] == "神奈川"]
```

</details>

---
## 9. 縦にくっつける (UNION ALL)

In [ ]:
jan = df[df["sale_date"] <= "2024-01-07"]
feb = df[df["sale_date"] > "2024-01-07"]
print(len(jan), len(feb))

both = pd.concat([jan, feb], ignore_index=True)
print(len(both))
both.head(3)

```sql
SELECT * FROM a UNION ALL SELECT * FROM b
```

```python
pd.concat([df1, df2, df3], ignore_index=True)
```

- **リストで渡します。** `df1.concat(df2)` とは書きません
- **列名で揃います。** 列の順番が違っても正しくくっつき、
  片方にしか無い列は欠損で埋まります
- `ignore_index=True` で index を振り直します。付けないと `0,1,2,0,1,2` のように重複します

複数ファイルを読んで1つにするときの定番です。

```python
frames = [pd.read_csv(p) for p in paths]
df = pd.concat(frames, ignore_index=True)
```

In [ ]:
# ミニ練習: 渋谷の行と横浜の行を、それぞれ絞ってから縦にくっつける
#           (index は振り直す)

shibuya = df[df["shop"] == "渋谷"]
yokohama = df[df["shop"] == "横浜"]

ans = ...   # ここに書く

assert len(ans) == 8, f"8行のはず: {len(ans)}"
assert ans.index.tolist() == list(range(8)), "index を振り直す"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.concat([shibuya, yokohama], ignore_index=True)
```

</details>

---
## 10. 重複を消す (DISTINCT)

In [ ]:
# 全列が一致する行を1つにする
print(len(both), "→", len(both.drop_duplicates()))

# キーを指定すると、その列が同じ行を1つにする
df[["shop", "item"]].drop_duplicates()

In [ ]:
# どれを残すかを選べる
d = pd.DataFrame({"k": ["a", "a", "b"], "v": [1, 2, 3]})
display(d)
display(d.drop_duplicates("k", keep="first"))   # 最初を残す (既定)
d.drop_duplicates("k", keep="last")             # 最後を残す

```sql
SELECT DISTINCT shop, item FROM sales
```

```python
df.drop_duplicates()                          # 全列一致
df.drop_duplicates(subset="key")              # キーで
df.drop_duplicates(subset="key", keep="last") # 最後を残す
```

「キーごとに最新の1行を残す」はこう書きます。並べてから、最後を残します。

```python
df.sort_values("updated_at").drop_duplicates("id", keep="last")
```

実務でとてもよく出る形です。詳しくは `pandas-03` でやります。

In [ ]:
# ミニ練習: shop の種類だけを、重複を消して取り出す (1列の表として)

ans = ...   # ここに書く

assert list(ans.columns) == ["shop"]
assert len(ans) == 3, f"3行のはず: {len(ans)}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[["shop"]].drop_duplicates()
```

</details>

---
## 11. 書く

In [ ]:
df.to_csv("/tmp/out.csv", index=False)
df.to_parquet("/tmp/out.parquet", index=False)

print(open("/tmp/out.csv").read()[:200])

```python
df.to_csv(path, index=False)
df.to_parquet(path, index=False)
```

**`index=False` を付けます。** 付けないと、意味のない連番が1列目に書き出されて、
次に読んだときに `Unnamed: 0` という謎の列になります。

保存は基本 Parquet です。CSV より小さく、速く、**型が保存されます**。
人が目で見るファイルだけ CSV にします。

In [ ]:
# ミニ練習: 渋谷の行だけを /tmp/shibuya.csv に書く (index は付けない)

shibuya = df[df["shop"] == "渋谷"]

# ここに書く

head = open("/tmp/shibuya.csv").read().split("\n")[0]
assert head.startswith("sale_id"), f"1列目が index になっている: {head}"
print("OK")

<details>
<summary>答え</summary>

```python
shibuya.to_csv("/tmp/shibuya.csv", index=False)
```

</details>

---
## つなげて書く

ここまでの操作はどれも**新しい DataFrame を返す**ので、`.` でつないで書けます。

In [ ]:
result = (
    df
    .query("qty >= 2")                                    # WHERE
    .groupby("shop", as_index=False)                      # GROUP BY
    .agg(n=("sale_id", "size"), total=("total", "sum"))   # 集約
    .sort_values("total", ascending=False)                # ORDER BY
    .reset_index(drop=True)
)
result

```sql
SELECT shop, count(*) AS n, sum(total) AS total
FROM sales WHERE qty >= 2
GROUP BY shop ORDER BY total DESC
```

SQL とほぼ同じ順に、上から読めます。

全体を括弧で囲むと、行末のバックスラッシュが要らなくなります。
つないだ途中で結果が変になったら、そこで切って `display()` してみると、
どこで想定と変わったかがすぐ分かります。

In [ ]:
# ミニ練習: 上の result の「絞り込み」を qty >= 3 に変えるだけ

ans = ...   # ここに書く

assert ans["shop"].tolist() == ["渋谷", "新宿"], ans["shop"].tolist()
assert ans["total"].tolist() == [3600, 1350]
print("OK")

<details>
<summary>答え</summary>

```python
ans = (
    df
    .query("qty >= 3")
    .groupby("shop", as_index=False)
    .agg(n=("sale_id", "size"), total=("total", "sum"))
    .sort_values("total", ascending=False)
    .reset_index(drop=True)
)
```

</details>

---
## 仕上げ

ここまでの10個を組み合わせるだけです。分からなければ答えを開いてかまいません。

In [ ]:
# 仕上げ1: item ごとの売上合計を出す。列名は item, total。合計の大きい順。

sales = pd.read_csv("/data/sales.csv")
sales["total"] = sales["qty"] * sales["price"]

ans = ...   # ここに書く

assert list(ans.columns) == ["item", "total"], list(ans.columns)
assert ans["item"].tolist() == ["コーヒー", "ケーキ", "紅茶"], ans["item"].tolist()
assert ans["total"].tolist() == [5400, 4200, 1600]
print("OK")

<details>
<summary>答え</summary>

```python
ans = (
    sales
    .groupby("item", as_index=False)
    .agg(total=("total", "sum"))
    .sort_values("total", ascending=False)
    .reset_index(drop=True)
)
```

</details>

In [ ]:
# 仕上げ2: 東京にある店の売上だけを合計する。(shops と結合してから絞る)

shops = pd.read_csv("/data/shops.csv")

ans = ...   # ここに書く

assert ans == 9150, f"9150 のはず: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
m = sales.merge(shops, on="shop", how="left")
ans = m[m["area"] == "東京"]["total"].sum()
```

</details>

In [ ]:
# 仕上げ3: 店ごとに、売上合計と、扱った商品の種類数を出す。
#          列名は shop, total, n_items。shop の昇順。

ans = ...   # ここに書く

assert list(ans.columns) == ["shop", "total", "n_items"], list(ans.columns)
assert ans["shop"].tolist() == ["新宿", "横浜", "渋谷"], ans["shop"].tolist()
assert ans["total"].tolist() == [3650, 2050, 5500]
assert ans["n_items"].tolist() == [3, 3, 3]
print("OK")

<details>
<summary>答え</summary>

```python
ans = (
    sales
    .groupby("shop", as_index=False)
    .agg(total=("total", "sum"), n_items=("item", "nunique"))
    .sort_values("shop")
    .reset_index(drop=True)
)
```

</details>

---
## 早見表

| # | やること | SQL | pandas |
| --- | --- | --- | --- |
| 1 | 読む | — | `pd.read_csv(path)` |
| 2 | 見る | — | `df.head()` / `df.shape` / `df.dtypes` / `df["c"].unique()` / `df["c"].value_counts()` |
| 3 | 列を選ぶ | `SELECT a, b` | `df[["a", "b"]]` |
| 4 | 行を絞る | `WHERE` | `df[(df["a"] == 1) & (df["b"] > 2)]` |
| 5 | 列を作る | `... AS x` | `df["x"] = df["a"] * df["b"]` |
| 6 | 並べる | `ORDER BY` | `df.sort_values("a", ascending=False)` |
| 7 | まとめる | `GROUP BY` | `df.groupby("k", as_index=False).agg(x=("a", "sum"))` |
| 8 | 横に結合 | `JOIN` | `df.merge(other, on="k", how="left")` |
| 9 | 縦に結合 | `UNION ALL` | `pd.concat([a, b], ignore_index=True)` |
| 10 | 重複を消す | `DISTINCT` | `df.drop_duplicates(subset="k", keep="last")` |
| — | 書く | — | `df.to_parquet(path, index=False)` |

### 忘れがちな3つ

- 条件は `&` `|` `~`。**それぞれ括弧で囲む**(`and` は使えない)
- `groupby` には `as_index=False`
- `to_csv` / `to_parquet` には `index=False`

### 数を数える癖

| 操作 | 確かめること |
| --- | --- |
| `merge` の後 | 行数が増えていないか、減っていないか |
| `groupby` の後 | 合計が元と一致するか |
| `drop_duplicates` の後 | 減った数に説明が付くか |
| 条件で絞った後 | 落ちた行が意図どおりか |

行数が変わる操作の前後で `len(df)` を見るだけで、事故の大半は防げます。

---

次は `pandas-01-read-and-types.ipynb` です。
今度は「引数を1つ変えると結果がどう変わるか」を見ていきます。